In [1]:
import os
import cv2
import torch
import numpy as np
import torch.nn as nn
import torchvision.models as models
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from tqdm import tqdm

# --- CONFIGURATION (UPDATE FF_PATH) ---
MODEL_WEIGHTS_PATH = "/kaggle/input/datasets/nithishkannaa/nonutdovemberlovember/best_dual_stream_model.pth"
FF_PATH = "/kaggle/input/datasets/debajyatidey/faceforensics-videos-cropped-faces" # <--- UPDATE THIS PATH

BATCH_SIZE = 64
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1. REBUILD ARCHITECTURE ---
class SRMConv2d(nn.Module):
    def __init__(self):
        super().__init__()
        srm_kernel = torch.tensor([
            [[-1,  2, -1],
             [ 2, -4,  2],
             [-1,  2, -1]]
        ], dtype=torch.float32).unsqueeze(1) / 4.0
        self.weight = nn.Parameter(srm_kernel.repeat(3, 1, 1, 1), requires_grad=False)
        
    def forward(self, x):
        return nn.functional.conv2d(x, self.weight, stride=1, padding=1, groups=3)

class DualStreamDeepfakeDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.rgb_stream = models.efficientnet_b4(weights=None)
        rgb_feature_dim = self.rgb_stream.classifier[1].in_features
        self.rgb_stream.classifier = nn.Identity()
        
        self.srm_filter = SRMConv2d()
        self.freq_stream = models.resnet18(weights=None)
        self.freq_stream.fc = nn.Identity()
        freq_feature_dim = 512
        
        self.fusion = nn.Sequential(
            nn.Linear(rgb_feature_dim + freq_feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 1)
        )
        
    def forward(self, x):
        spatial_features = self.rgb_stream(x)
        noise_x = self.srm_filter(x)
        freq_features = self.freq_stream(noise_x)
        combined = torch.cat((spatial_features, freq_features), dim=1)
        return self.fusion(combined)

# --- 2. LOAD & LOCK WEIGHTS ---
print("Initializing model architecture...")
model = DualStreamDeepfakeDetector()

print(f"Loading trained weights from: {MODEL_WEIGHTS_PATH}")
model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=device))
model.to(device)
model.eval()

# --- 3. DATA LOADER ---
class FaceForensicsEvaluationDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.image_paths = []
        self.labels = []
        
        valid_extensions = {".jpg", ".jpeg", ".png"}
        all_images = [p for p in self.root_dir.rglob("*") if p.suffix.lower() in valid_extensions]
        
        for img_path in all_images:
            path_str = str(img_path).lower()
            
            # Map based on folder names "real" and "fake"
            if "/real/" in path_str or "\\real\\" in path_str:
                self.labels.append(0)
                self.image_paths.append(img_path)
            elif "/fake/" in path_str or "\\fake\\" in path_str:
                self.labels.append(1)
                self.image_paths.append(img_path)
                
        print(f"FF++ (C23) Benchmark Pool Ready: {len(self.image_paths)} total images loaded.")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = str(self.image_paths[idx])
        label = self.labels[idx]
        
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if self.transform:
            image = self.transform(image=image)['image']
            
        return image, label

eval_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

ff_dataset = FaceForensicsEvaluationDataset(root_dir=FF_PATH, transform=eval_transform)
ff_loader = DataLoader(ff_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# --- 4. INFERENCE LOOP ---
print("\nStarting Cross-Dataset Evaluation on FaceForensics++ (C23)...")
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(ff_loader, desc="Running Inference"):
        images = images.to(device)
        outputs = model(images).squeeze(1)
        probs = torch.sigmoid(outputs).cpu().numpy()
        
        all_preds.extend(probs)
        all_labels.extend(labels.numpy())

# --- 5. CALCULATE METRICS ---
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

binary_preds = (all_preds > 0.5).astype(int)

accuracy = accuracy_score(all_labels, binary_preds)
auc_roc = roc_auc_score(all_labels, all_preds)

print("\n" + "="*40)
print("FINAL FACEFORENSICS++ (C23) BENCHMARK RESULTS")
print("="*40)
print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"AUC-ROC:  {auc_roc:.4f}")
print("="*40)
print("\nDetailed Classification Report:")
print(classification_report(all_labels, binary_preds, target_names=["Real (0)", "Fake (1)"]))

Initializing model architecture...
Loading trained weights from: /kaggle/input/datasets/nithishkannaa/nonutdovemberlovember/best_dual_stream_model.pth
FF++ (C23) Benchmark Pool Ready: 39949 total images loaded.

Starting Cross-Dataset Evaluation on FaceForensics++ (C23)...


Running Inference: 100%|██████████| 625/625 [1:48:38<00:00, 10.43s/it]


FINAL FACEFORENSICS++ (C23) BENCHMARK RESULTS
Accuracy: 64.11%
AUC-ROC:  0.7025

Detailed Classification Report:
              precision    recall  f1-score   support

    Real (0)       0.67      0.55      0.61     19996
    Fake (1)       0.62      0.73      0.67     19953

    accuracy                           0.64     39949
   macro avg       0.65      0.64      0.64     39949
weighted avg       0.65      0.64      0.64     39949

